In [ ]:
# =========================================================
# CONVERSIÓN A FOA DESDE:
#   1) AUDIO ESTÉREO (2 canales)
#   2) 4 MICRÓFONOS CARDIOIDES TETRAÉDRICOS (4 canales)
#
# Salidas:
#   - output_foa.wav        -> FOA en orden ambiX [W, Y, Z, X]
#   - output_binaural.wav   -> render binaural aproximado
#
# NOTA:
# - Modo estéreo: usa estimación 3D coherente con el análisis del libro
# - Modo 4 canales: usa matriz tetraédrica tipo FLU, FRD, BLD, BRU
# =========================================================

In [ ]:
# =========================================================
# INSTALACIÓN
# =========================================================
!pip install -q soundfile scipy pysofaconventions
!apt-get install -y ffmpeg curl > /dev/null

# =========================================================
# IMPORTACIÓN
# =========================================================
import subprocess
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

from scipy.signal import stft, istft, fftconvolve
from pysofaconventions import SOFAFile
from google.colab import files
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
# =========================================================
# PARÁMETROS GENERALES DSP (v4.0)
# =========================================================
N_FFT: int = 2048  # Tamaño del análisis en frecuencia
HOP: int = N_FFT // 4 # Solapamiento (Overlap = N_FFT - HOP)

# Parámetros del modo estéreo -> FOA
MAX_ELEV_DEG: float = 45.0 # Elevación máxima realista
ALPHA_AZ: float = 0.20 # Filtros de suavizado (azimut)
ALPHA_EL: float = 0.85 # Filtros de suavizado (elevación)

# Render binaural desde FOA (Altavoces virtuales AmbiX)
VIRTUAL_SPEAKERS = [
    (0.0,    0.0, 1.00),   # frente
    (60.0,   0.0, 0.90),   # izquierda-frente
    (-60.0,  0.0, 0.90),   # derecha-frente
    (180.0,  0.0, 0.55),   # atrás
    (0.0,   55.0, 0.80),   # arriba más perceptible
    (0.0,  -45.0, 0.35),   # abajo
]


In [ ]:
# =========================================================
# FUNCIONES AUXILIARES DSP
# =========================================================
import numpy as np

def rms(x: np.ndarray) -> float:
    """Calcula la energía promedio (RMS) de una señal."""
    return float(np.sqrt(np.mean(x**2) + 1e-12))

def smooth1d(y: np.ndarray, win: int = 9) -> np.ndarray:
    """Suaviza señales 1D usando una ventana de media móvil."""
    y = np.asarray(y, dtype=float)
    if win < 2 or len(y) < win:
        return y.copy()
    if win % 2 == 0:
        win += 1
    k = np.ones(win) / win
    pad = win // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(ypad, k, mode="valid")

def wrap_deg(x: np.ndarray) -> np.ndarray:
    """Envuelve grados al rango [-180, 180]."""
    return ((x + 180) % 360) - 180


In [ ]:
# Celda limpia por Refactorización v4.0
# Se eliminaron las funciones de graficación y análisis de visualización (código muerto)
# para garantizar máxima eficiencia y RTF < 0.60 en el entorno de producción.


In [ ]:
# =========================================================
# HRTF
# =========================================================
def load_sofa(path): #formato estándar para guardar: htrf, posiciones espaciale, respuestas impulsionales, cómo escucha la cabeza humana desde muchas direcciones
    s = SOFAFile(path, 'r') #r : leer, no modificar, abre el archivo SOFA
    ir = s.getDataIR() #impulse Response: cómo responde el oído a un sonido desde cierta dirección - ir[posición, oído, muestras]
    ''' oído
            L        R
pos 0   [1 2 3]  [4 5 6]

pos 1   [7 8 9]  [10 11 12]

pos 2   [13 14 15] [16 17 18]'''

    pos = s.getVariableValue('SourcePosition') #búscame una variable dentro del archivo SOFA - Las direcciones donde fueron medidas las HRTFs - [azimut, elevación, distancia]
    return ir, pos

def get_hrtf_interp(az_deg, el_deg, hrtf, pos, k=4): #compara con posiciones reales,busca las más cercanas, hace promedio - función recibe un ángulo deseado
    az_deg = wrap_deg(az_deg)

    az_all = wrap_deg(pos[:, 0]) #toma TODOS los azimuts del SOFA y los normaliza
    el_all = pos[:, 1]

    az_diff = np.abs(wrap_deg(az_all - az_deg)) #diferencia entre ángulo real y ángulo deseado - distancia angular horizontal - qué tan cerca está cada HRTF -pequeña diferencia = buena HRTF candidata
    el_diff = np.abs(el_all - el_deg)

    dist = np.sqrt((az_diff / 180.0)**2 + (el_diff / 90.0)**2) #calcula distancia espacial aproximada - pitágoras - qué tan lejos está cada HRTF de la dirección deseada
    idx = np.argsort(dist)[:k] #ordena índices de menor a mayor - es un vector de índices - tomar los primeros k elementos

    w = 1.0 / (dist[idx] + 1e-6) #crea pesos- HRTF más cercana = más importante.
    w /= np.sum(w) #normaliza pesos - w = w / suma_total

    hL = np.sum(hrtf[idx, 0, :] * w[:, None], axis=0)  #Hace la interpolación - las HRTFs izquierdas más cercanas, none pasa de fila a columna para poder multiplicar fila a fila - Porque las dimensiones no coinciden bien para broadcasting.
    hR = np.sum(hrtf[idx, 1, :] * w[:, None], axis=0)

    return hL, hR #filtro izquierdo final, filtro derecho final


In [ ]:
# =========================================================
# CARGA DE AUDIO
# =========================================================
def upload_and_load_audio():
    uploaded = files.upload()  #abre el selector de archivos en Colab, queda siendo un diccionario.
    audio_file = list(uploaded.keys())[0] #toma el nombre del archivo

    if audio_file.lower().endswith(".mp3"): #convierte a minúsculas, toma el audio,conviértelo a WAV sin compresión,a 44100 Hz,y guárdalo como converted.wav
        subprocess.run([ #ejecuta un comando externo ffmpg convierte audio/video
            "ffmpeg",#Convertir mp3 → wav.
            "-y", # son comandos - sobrescribir sin preguntar
            "-loglevel", "quiet", #no mostrar mensajes en consola
            "-i", audio_file, #input (archivo de entrada)
            "-acodec", "pcm_s16le", #define formato WAV PCM 16 bits, audio SIN compresión
            "-ar", "44100", #resamplea a 44100 Hz, estándar de audio
            "converted.wav" #nombre del archivo de salida
        ], check=True)
        audio_file = "converted.wav" #el WAV convertido

    audio, sr = sf.read(audio_file) #carga el audio en memoria, audio contiene las muestras, sr, sample rate 44100, lee el archivo de audio y lo convierte en números
    return audio, sr, audio_file

In [ ]:
# =========================================================
# PREPROCESAMIENTO
# =========================================================
def normalize_multichannel(audio): #( N, canales ) limpio, centrado y sin saturación
    audio = audio.astype(np.float64) #los convierte en float
    if audio.ndim == 1:
        audio = audio[:, None] #Para que TODO el código funcione igual para mono y estéreo. (N, canales)

    # quitar DC por canal
    audio = audio - np.mean(audio, axis=0, keepdims=True) #promedio por columna (por canal) (1,2) mantén la forma compatible

    peak = np.max(np.abs(audio)) + 1e-9 #Busca el pico, el valor mas grande
    audio = 0.95 * audio / peak #normaliza
    return audio

In [ ]:
# =========================================================
# MODO 1: ESTÉREO -> FOA 3D ESTIMADO
# Salida compatible con AmbiX ACN/SN3D
# =========================================================

from scipy.ndimage import gaussian_filter1d

def stereo_to_foa(audio_stereo, sr):
    """
    Convierte audio estéreo L/R a una representación FOA 3D estimada.

    Importante:
    - El audio estéreo original NO es ambisónico.
    - El código estima una dirección espacial a partir de balance L/R,
      coherencia, difusión, ILD y contenido en altas frecuencias.
    - La salida se construye como FOA compatible con AmbiX ACN/SN3D.
    """

    # -----------------------------
    # SEPARAR CANALES
    # -----------------------------
    audio_stereo = np.asarray(audio_stereo, dtype=np.float64)

    if audio_stereo.ndim != 2 or audio_stereo.shape[1] < 2:
        raise ValueError("El modo estéreo requiere un audio de mínimo 2 canales: L y R.")

    L = audio_stereo[:, 0]
    R = audio_stereo[:, 1]

    # -----------------------------
    # STFT
    # -----------------------------
    f, t, ZL = stft(L, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)
    _, _, ZR = stft(R, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)

    eps = 1e-12
    magL = np.abs(ZL)
    magR = np.abs(ZR)

    # Mid / Side
    M = (ZL + ZR) / 2.0
    S = (ZL - ZR) / 2.0

    # Energía base
    E = np.abs(M) ** 2

    # =====================================================
    # AZIMUT ESTIMADO
    # =====================================================
    balance = (magL - magR) / (magL + magR + eps)
    balance = np.clip(balance, -1.0, 1.0)

    az = np.arcsin(balance)
    az = np.clip(az, -np.pi / 2, np.pi / 2)

    # Suavizado temporal del azimut por frecuencia
    for k in range(1, az.shape[1]):
        az[:, k] = ALPHA_AZ * az[:, k - 1] + (1 - ALPHA_AZ) * az[:, k]

    # Azimut promedio por trama, ponderado por energía
    az_frame = np.sum(az * E, axis=0) / (np.sum(E, axis=0) + eps)
    az_frame = np.clip(az_frame, -np.pi / 2, np.pi / 2)

    # =====================================================
    # ELEVACIÓN MEJORADA
    # =====================================================

    # Coherencia intercanal, qué tan parecidos son L y R
    coh = np.abs(ZL * np.conj(ZR)) / (magL * magR + eps)
    coh = np.clip(coh, 0.0, 1.0)

    # Frecuencias altas para estimación vertical, f Vector de frecuencias del STFT.
    hf_mask = f >= 3000.0

    # Seguridad por si el audio tiene frecuencia de muestreo baja
    if not np.any(hf_mask):
        hf_mask = f >= 0.5 * np.max(f)

    E_hf = E[hf_mask, :] #todas las frecuencias altas, para todos los tiempos

    # Difusión en altas frecuencias
    coh_hf = np.sum(coh[hf_mask, :] * E_hf, axis=0) / (np.sum(E_hf, axis=0) + eps) #qué tan parecidos son los agudos importantes
    diffuse_hf = 1.0 - coh_hf
    diffuse_hf = np.clip(diffuse_hf, 0.0, 1.0)

    # ILD en altas frecuencias, diferencia de volumen entre oídos, Interaural Level Difference, El cerebro usa eso para dirección.
    ild = (magL - magR) / (magL + magR + eps)
    ild_hf = np.sum(np.abs(ild[hf_mask, :]) * E_hf, axis=0) / (np.sum(E_hf, axis=0) + eps) #qué tan lateralizados están los agudos
    ild_hf = np.clip(ild_hf, 0.0, 1.0)

    # Tilt espectral / centroide normalizado, dónde está concentrada la energía, grave - bajo, agudo - alto
    spectral_tilt = np.sum(E * f[:, None], axis=0) / (np.sum(E, axis=0) + eps)
    spectral_tilt = (
        spectral_tilt - np.min(spectral_tilt)
    ) / (
        np.max(spectral_tilt) - np.min(spectral_tilt) + eps
    )
    spectral_tilt = np.clip(spectral_tilt, 0.0, 1.0)

    # Confianza vertical, voy a inventarme una probabilidad de que el sonido esté arriba
    z_conf = (
        0.4 * spectral_tilt +
        0.3 * diffuse_hf +
        0.3 * ild_hf
    )

    # Reducir elevación cuando el sonido está muy lateralizado
    z_conf *= np.cos(az_frame)

    # Asegurar rango válido
    z_conf = np.clip(z_conf, 0.0, 1.0)

    # Suavizado de la confianza vertical, quita cambios bruscos, sigma controla cuánto suaviza.
    z_conf = gaussian_filter1d(z_conf, sigma=2)

    # Elevación final por trama,            Potencia
    el_frame = np.deg2rad(MAX_ELEV_DEG) * (z_conf ** 0.7) #para exagerar un poco, elevación humana es poco sensible.

    # Suavizado temporal de elevación, no cambies elevación bruscament
    for k in range(1, len(el_frame)):
        el_frame[k] = ALPHA_EL * el_frame[k - 1] + (1 - ALPHA_EL) * el_frame[k]

    # Evitar artefactos iniciales, Infinite Impulse Response, ELEGIR VALOR ESTABLE
    if len(el_frame) > 20:
        el_frame[:20] = el_frame[20]

    # Expandir elevación a todos los bits de frecuencia
    el = np.tile(el_frame[None, :], (len(f), 1))

    # =====================================================
    # VECTOR DIRECCIÓN 3D
    # Convención:
    # +X = frente
    # +Y = izquierda
    # +Z = arriba
    # ángulos → coordenadas 3D
    # =====================================================
    theta_z = np.sin(el)
    theta_h = np.cos(el)
    theta_x = theta_h * np.cos(az)
    theta_y = theta_h * np.sin(az)

    # =====================================================
    # CONSTRUCCIÓN FOA - BASE AmbiX ACN/SN3D
    # =====================================================
    # En SN3D de primer orden:
    # W = señal omnidireccional
    # X = señal * componente frontal/trasera
    # Y = señal * componente izquierda/derecha
    # Z = señal * componente arriba/abajo
    #
    # Aquí NO se usa M/sqrt(2), ni X_GAIN, Y_GAIN, Z_GAIN,
    # para no alterar la normalización relativa SN3D.

    W_tf = M
    X_tf = M * theta_x
    Y_tf = M * theta_y
    Z_tf = M * theta_z

    # =====================================================
    # VOLVER AL DOMINIO DEL TIEMPO
    # =====================================================
    _, W = istft(W_tf, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)
    _, X = istft(X_tf, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)
    _, Y = istft(Y_tf, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)
    _, Z = istft(Z_tf, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP)

    # Igualar longitudes (Por el solapamiento de ventanas la ISTFT puede devolver
    # arrays con longitudes ligeramente diferentes. Esta parte simplemente los
    # recorta todos al mismo tamaño para que puedan empacarse juntos sin problemas.)
    N = min(len(W), len(X), len(Y), len(Z), len(L), len(R))
    W = W[:N]
    X = X[:N]
    Y = Y[:N]
    Z = Z[:N]

    diagnostics = {
        "f": f,
        "t": t,
        "ZL": ZL,
        "ZR": ZR,
        "M": M,
        "S": S,
        "az_frame_deg": smooth1d(np.rad2deg(az_frame), 9),
        "el_frame_deg": smooth1d(np.rad2deg(el_frame), 11),
        "z_conf": z_conf,
        "normalization": "AmbiX ACN/SN3D compatible",
        "input_mode": "stereo_to_estimated_foa"
    }

    return W, X, Y, Z, diagnostics

In [ ]:
# =========================================================
# MODO 2: 4 MICRÓFONOS TETRAÉDRICOS -> FOA 3D
# Salida compatible con AmbiX ACN/SN3D
# =========================================================

def tetra_aformat_to_foa(audio4):
    """
    Convierte A-format tetraédrico idealizado a FOA 3D.

    Orden esperado de entrada:
    canal 0 = FLU
    canal 1 = FRD
    canal 2 = BLD
    canal 3 = BRU

    Salida interna:
    W, X, Y, Z

    Luego, al guardar, se debe ordenar como AmbiX/ACN:
    [W, Y, Z, X]
    """

    audio4 = np.asarray(audio4, dtype=np.float64)

    if audio4.ndim != 2 or audio4.shape[1] < 4:
        raise ValueError("Se requieren 4 canales en orden FLU, FRD, BLD, BRU.")

    # Matriz tetraédrica compatible con FOA/SN3D
    M_tetra = 0.5 * np.array([
        [1.0,        1.0,        1.0,        1.0],
        [np.sqrt(3), np.sqrt(3), -np.sqrt(3), -np.sqrt(3)],
        [np.sqrt(3), -np.sqrt(3), np.sqrt(3), -np.sqrt(3)],
        [np.sqrt(3), -np.sqrt(3), -np.sqrt(3), np.sqrt(3)],
    ], dtype=np.float64)

    WXYZ = audio4[:, :4] @ M_tetra.T

    W = WXYZ[:, 0]
    X = WXYZ[:, 1]
    Y = WXYZ[:, 2]
    Z = WXYZ[:, 3]

    diagnostics = {
        "encoder_matrix": M_tetra,
        "normalization": "AmbiX ACN/SN3D compatible",
        "input_mode": "tetra_aformat_to_foa",
        "channel_order_input": "FLU, FRD, BLD, BRU",
        "channel_order_internal": "W, X, Y, Z"
    }

    return W, X, Y, Z, diagnostics

In [ ]:
# =========================================================
# FOA -> BINAURAL Y PROCESAMIENTO DINÁMICO
# =========================================================
def foa_to_binaural(W, X, Y, Z, sr, hrtf, pos):
    N = min(len(W), len(X), len(Y), len(Z))
    W = W[:N]; X = X[:N]; Y = Y[:N]; Z = Z[:N]

    outL = np.zeros(N + 512)
    outR = np.zeros(N + 512)

    for az_deg, el_deg, gain in VIRTUAL_SPEAKERS:
        az = np.deg2rad(az_deg)
        el = np.deg2rad(el_deg)

        x = np.cos(el) * np.cos(az)
        y = np.cos(el) * np.sin(az)
        z = np.sin(el)

        spk = gain * ((W / np.sqrt(2)) + x * X + y * Y + z * Z)
        hL, hR = get_hrtf_interp(az_deg, el_deg, hrtf, pos, k=4)

        convL = fftconvolve(spk, hL, mode="full")
        convR = fftconvolve(spk, hR, mode="full")

        outL[:len(convL)] += convL
        outR[:len(convR)] += convR

    N2 = min(len(outL), len(outR))
    binaural = np.stack([outL[:N2], outR[:N2]], axis=1)
    binaural /= np.max(np.abs(binaural)) + 1e-9
    return binaural

def foa_motion_preview(W, X, Y, Z, sr, rate_az=0.12, rate_el=0.09, max_el_deg=58.0, z_preview_gain=3):
    N = min(len(W), len(X), len(Y), len(Z))
    W = W[:N]; X = X[:N]; Y = Y[:N]; Z = Z[:N]
    t = np.arange(N) / sr

    az = 2 * np.pi * rate_az * t
    X_rot = np.cos(az) * X - np.sin(az) * Y
    Y_rot = np.sin(az) * X + np.cos(az) * Y

    el = np.deg2rad(max_el_deg) * np.sin(2 * np.pi * rate_el * t)
    X_3d = np.cos(el) * X_rot - np.sin(el) * (z_preview_gain * Z)
    Z_3d = np.sin(el) * X_rot + np.cos(el) * (z_preview_gain * Z)

    return W, X_3d, Y_rot, Z_3d

def normalize_audio(x, peak_target=0.95):
    peak = np.max(np.abs(x)) + 1e-9
    if peak > 0:
        x = peak_target * x / peak
    return x

# Calibración Oficial v4.0 (Obligatoria para paridad con Backend Web)
def stabilize_loudness(y, sr,
                       win_ms=455,
                       strength=0.70,
                       min_gain=0.88,
                       max_gain=2.0):
    eps = 1e-9
    y = np.asarray(y, dtype=np.float64)
    power = np.mean(y**2, axis=1)

    win = int(sr * win_ms / 1000)
    win = max(win, 1)
    kernel = np.ones(win) / win

    env = np.sqrt(np.convolve(power, kernel, mode="same") + eps)
    target = np.median(env) + eps

    gain = target / (env + eps)
    gain = gain ** strength
    gain = np.clip(gain, min_gain, max_gain)
    gain = np.convolve(gain, kernel, mode="same")

    y_out = y * gain[:, None]
    return normalize_audio(y_out, peak_target=0.95)


In [ ]:
# =========================================================
# DESCARGA HRTF SOFA (CIPIC Subject 003)
# =========================================================

!wget -O hrtf.sofa "https://raw.githubusercontent.com/hoene/libmysofa/main/tests/CIPIC_subject_003_hrir_final_itdInDelayField.sofa"

# =========================================================
# CARGAR HRTF
# =========================================================

hrtf, pos = load_sofa("hrtf.sofa")

print("HRTF cargado correctamente")
print("Shape HRTF:", hrtf.shape)

if pos is not None:
    print("Shape posiciones:", pos.shape)

--2026-11-14 16:54:47--  https://raw.githubusercontent.com/hoene/libmysofa/main/tests/CIPIC_subject_003_hrir_final_itdInDelayField.sofa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3377059 (3.2M) [application/octet-stream]
Saving to: ‘hrtf.sofa’

hrtf.sofa           100%[===================>]   3.22M  --.-KB/s    in 0.07s   

2026-11-14 16:54:48 (46.4 MB/s) - ‘hrtf.sofa’ saved [3377059/3377059]

HRTF cargado correctamente
Shape HRTF: (1250, 2, 200)
Shape posiciones: (1250, 3)


In [ ]:
# =========================================================
# CARGAR AUDIO Y VERIFICAR RESTRICCIONES V4.0
# =========================================================
audio, sr, audio_file = upload_and_load_audio()

# Validación oficial v4.0: Rechazar frecuencias menores a 44.1 kHz
if sr < 44100:
    raise ValueError(f"Error de cumplimiento: Frecuencia de muestreo ({sr} Hz) inferior a 44100 Hz. Rechazado.")

audio = normalize_multichannel(audio)

if audio.ndim == 1:
    audio = audio[:, None]

num_ch = audio.shape[1]
print(f"Archivo cargado: {audio_file}")
print(f"Frecuencia de muestreo: {sr} Hz (Válida)")
print(f"Número de canales detectados: {num_ch}")


Saving y2mate.com - Diomedes Diaz y Joe Arroyo - Ron Pa´ Todo El Mundo.mp3 to y2mate.com - Diomedes Diaz y Joe Arroyo - Ron Pa´ Todo El Mundo.mp3
Archivo cargado: converted.wav
Frecuencia de muestreo: 44100 Hz (Válida)
Número de canales detectados: 2


In [ ]:
# =========================================================
# CONFIGURACIÓN Y SELECCIÓN DE MODO AUTOMATIZADA
# =========================================================
# Centralización de parámetros para ejecución continua sin intervención manual.
CONFIG = {
    "mode": "auto",           # Opciones: "auto", "stereo", "tetra_4mic"
    "tetra_order": [0, 1, 2, 3] # Orden de canales: FLU, FRD, BLD, BRU
}

print("\n===== SELECCIÓN DE MODO =====")
mode = CONFIG["mode"]

try:
    if mode == "auto":
        if 'num_ch' not in globals():
            print("Advertencia: 'num_ch' no está definido. Asegúrese de ejecutar la celda de carga de audio. Usando fallback estéreo.")
            mode = "stereo"
        elif num_ch >= 4:
            mode = "tetra_4mic"
        elif num_ch == 2:
            mode = "stereo"
        else:
            raise ValueError("Se requiere un archivo estéreo (2 canales) o tetra (4 canales).")

    print(f"\nModo final usado: {mode}")

    if mode == "stereo":
        if 'num_ch' in globals() and num_ch < 2:
            raise ValueError("Para modo estéreo se necesitan al menos 2 canales.")
        if 'audio' in globals() and 'sr' in globals():
            W, X, Y, Z, diagnostics = stereo_to_foa(audio[:, :2], sr)
        else:
            print("Advertencia: El audio no ha sido cargado en memoria.")

    elif mode == "tetra_4mic":
        if 'num_ch' in globals() and num_ch < 4:
            raise ValueError("Para modo tetra_4mic se necesitan al menos 4 canales.")

        order = CONFIG["tetra_order"]
        if 'audio' in globals():
            audio4 = audio[:, order]
            print(f"Orden usado (0-based): {order}")
            W, X, Y, Z, diagnostics = tetra_aformat_to_foa(audio4)
        else:
            print("Advertencia: El audio no ha sido cargado en memoria.")

    else:
        if mode not in ["stereo", "tetra_4mic"]:
            raise ValueError("Modo no reconocido.")

except Exception as e:
    print(f"Error durante la selección de modo: {e}")


In [ ]:
# =========================================================
# NORMALIZAR FOA Y EMPACAR EN AMBIX [W, Y, Z, X]
# =========================================================
if 'W' in globals() and 'X' in globals() and 'Y' in globals() and 'Z' in globals():
    N = min(len(W), len(X), len(Y), len(Z))
    W = W[:N]
    X = X[:N]
    Y = Y[:N]
    Z = Z[:N]

    foa = np.stack([W, Y, Z, X], axis=1)
    foa /= np.max(np.abs(foa)) + 1e-9

    print("\n===== VALIDACIÓN FOA =====")
    print("Shape FOA:", foa.shape)
    print(f"RMS W: {rms(W):.6f}")
    print(f"RMS X: {rms(X):.6f}")
    print(f"RMS Y: {rms(Y):.6f}")
    print(f"RMS Z: {rms(Z):.6f}")
else:
    print("Error: Las variables FOA (W, X, Y, Z) no han sido generadas.")
    print("Por favor, cargue el audio y ejecute el motor de codificación previamente.")



===== VALIDACIÓN FOA =====
Shape FOA: (11366400, 4)
RMS W: 0.116072
RMS X: 0.112869
RMS Y: 0.016112
RMS Z: 0.020817


In [ ]:
# =========================================================
# RENDER BINAURAL (v4.0)
# =========================================================

# 1) Binaural normal desde el FOA limpio
binaural = foa_to_binaural(W, X, Y, Z, sr, hrtf, pos)
binaural = normalize_audio(binaural, peak_target=0.95)

# 2) Binaural 3D perceptual con movimiento ambisónico
W_3d, X_3d, Y_3d, Z_3d = foa_motion_preview(
    W, X, Y, Z, sr,
    rate_az=0.12,
    rate_el=0.10,
    max_el_deg=60.0,
    z_preview_gain=3.5
)

binaural_3d = foa_to_binaural(W_3d, X_3d, Y_3d, Z_3d, sr, hrtf, pos)
binaural_3d = 0.90 * binaural_3d + 0.10 * binaural

# Estabilizar volumen usando los parámetros oficiales de calibración (v4.0)
binaural_3d = stabilize_loudness(
    binaural_3d,
    sr,
    win_ms=455,
    strength=0.70,
    min_gain=0.88,
    max_gain=2.0
)

# 3) Validación de integridad
ild_out = 20 * np.log10((rms(binaural[:, 0]) + 1e-12) / (rms(binaural[:, 1]) + 1e-12))
ild_3d = 20 * np.log10((rms(binaural_3d[:, 0]) + 1e-12) / (rms(binaural_3d[:, 1]) + 1e-12))

print("\n===== VALIDACIÓN BINAURAL NORMAL =====")
print(f"RMS L: {rms(binaural[:,0]):.6f} | RMS R: {rms(binaural[:,1]):.6f} | ILD: {ild_out:.2f} dB")

print("\n===== VALIDACIÓN BINAURAL 3D PERCEPTUAL =====")
print(f"RMS L: {rms(binaural_3d[:,0]):.6f} | RMS R: {rms(binaural_3d[:,1]):.6f} | ILD: {ild_3d:.2f} dB")


KeyboardInterrupt: 

In [ ]:
from IPython.display import Audio

# =========================================================
# GUARDAR RESULTADOS
# =========================================================

# ---------------------------------------------------------
# 1) Guardar FOA limpio
# ---------------------------------------------------------
# Este es el archivo principal del proyecto.
# Orden AmbiX / ACN:
# Canal 0 = W
# Canal 1 = Y
# Canal 2 = Z
# Canal 3 = X

sf.write("output_foa.wav", foa, sr)


# ---------------------------------------------------------
# 2) Guardar binaural normal
# ---------------------------------------------------------
sf.write("output_binaural.wav", binaural, sr, subtype="PCM_16")

# MP3 opcional para mayor compatibilidad
subprocess.run([
    "ffmpeg", "-y",
    "-i", "output_binaural.wav",
    "-codec:a", "libmp3lame",
    "-b:a", "320k",
    "output_binaural.mp3"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ---------------------------------------------------------
# 3) Guardar binaural 3D perceptual
# ---------------------------------------------------------
sf.write("output_binaural_3D_perceptual.wav", binaural_3d, sr, subtype="PCM_16")

# MP3 opcional para mayor compatibilidad
subprocess.run([
    "ffmpeg", "-y",
    "-i", "output_binaural_3D_perceptual.wav",
    "-codec:a", "libmp3lame",
    "-b:a", "320k",
    "output_binaural_3D_perceptual.mp3"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)



# =========================================================
# REPRODUCIR EN COLAB
# =========================================================

print("\n🎧 BINAURAL NORMAL")
display(Audio("output_binaural.wav"))

print("\n🎧 BINAURAL 3D PERCEPTUAL")
display(Audio("output_binaural_3D_perceptual.wav"))


# =========================================================
# DESCARGAR ARCHIVOS
# =========================================================

from IPython.display import HTML, display
import base64

def make_download_link(path, label, color="#6d28d9"):
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return f'''
    <a download="{path}" href="data:application/octet-stream;base64,{b64}"
       style="
           display:inline-block;
           margin-right:12px;
           margin-top:8px;
           margin-bottom:8px;
           padding:10px 16px;
           background:{color};
           color:white;
           text-decoration:none;
           border-radius:10px;
           font-weight:600;
       ">
       {label}
    </a>
    '''

display(HTML("<h3>Opciones de descarga</h3>"))

display(HTML("""
<p><b>WAV (mejor calidad):</b> recomendado para escuchar en computador o en teléfonos con reproductores compatibles, como VLC.</p>
<p><b>MP3 (mayor compatibilidad):</b> recomendado para escuchar fácilmente en teléfono móvil, compartir por aplicaciones como WhatsApp y reproducir en la mayoría de dispositivos.</p>
"""))

# FOA
display(HTML("<h4>FOA</h4>"))
display(HTML(
    make_download_link("output_foa.wav", "Descargar FOA WAV", "#4c1d95")
))

# Binaural
display(HTML("<h4>Binaural</h4>"))
display(HTML(
    make_download_link("output_binaural.wav", "Descargar binaural WAV", "#6d28d9") +
    make_download_link("output_binaural.mp3", "Descargar binaural MP3", "#9333ea")
))

# 3D perceptual
display(HTML("<h4>3D perceptual</h4>"))
display(HTML(
    make_download_link("output_binaural_3D_perceptual.wav", "Descargar 3D perceptual WAV", "#2563eb") +
    make_download_link("output_binaural_3D_perceptual.mp3", "Descargar 3D perceptual MP3", "#0ea5e9")
))


# =========================================================
# MENSAJES FINALES
# =========================================================

print("\nPROCESO COMPLETADO")
print("Modo usado:", mode)

print("\nSalida FOA limpia:")
print("Archivo: output_foa.wav")
print("Orden AmbiX / ACN: [W, Y, Z, X]")

print("\nSalida binaural normal:")
print("Archivo: output_binaural.wav")

print("\nSalida binaural 3D perceptual:")
print("Archivo: output_binaural_3D_perceptual.wav")
print("Movimiento aplicado sobre canales ambisónicos W, X, Y, Z.")



🎧 BINAURAL NORMAL
Buffered data was truncated after reaching the output size limit.